# Modelos avançados no Kaggle

Este notebook **não roda no ambiente local** — foi gerado aqui, mas é
feito para ser subido e executado no Kaggle, onde PyTorch + GPU já vêm
prontos (ver `reports/tables/modelos_avancados_smoke_test.md` para o
porquê de não dar para rodar isso localmente).

Segue a mesma organização dos outros notebooks deste repositório: o código
fica no pacote `src/churn_telecom/`, e este notebook só clona o repositório,
instala o pacote e orquestra os experimentos.

**No Kaggle, antes de rodar:** ative *Settings → Internet → On* (o
notebook clona o repositório, baixa o CSV e instala os pacotes) e
*Accelerator → GPU*. Depois use **"Save Version" → "Save & Run All
(Commit)"**, que executa num kernel gerenciado em segundo plano.


In [ ]:
#@title Setup (clona o repo e instala o pacote)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"   # ajuste se necessario
BRANCH   = "feat/churn"   # ajuste aqui se a branch de trabalho mudar
REPO_DIR = "redes-neurais"
SUBPATH  = "projeto/churn-telecom"

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    base = Path("/content") if IN_COLAB else Path("/kaggle/working")
    root = base / REPO_DIR
    if root.exists():
        _sh(f"cd {root} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
    else:
        _sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {root}")
    PROJECT_ROOT = root / SUBPATH
    _sh(f"pip install -q -e {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR    = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR  = PROJECT_ROOT / "reports" / "tables"
for d in (DATA_DIR, RESULTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)


## Instalação dos modelos avançados

O Kaggle já vem com PyTorch e a maioria das libs de ML — só falta instalar
os 5 modelos em si.

In [ ]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


## Preparação dos dados

`load_raw` baixa o CSV automaticamente se não existir em `DATA_DIR`
(mesma conveniência do mini-projeto 2) — não é preciso subir nada como
Kaggle Dataset separado, o clone do repositório já basta.

In [ ]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")


## Execução isolada por modelo

Cada modelo roda dentro de `rodar(nome, fn)`: se um falhar (pacote,
download de pesos, memória...), o erro é registrado em `FALHAS` e o
notebook **continua** para os próximos — um "Save & Run All" nunca mais
morre no meio por causa de um único modelo. As falhas são salvas em
`reports/tables/modelos_avancados_falhas_kaggle.csv` no final.

In [ ]:
import traceback

FALHAS = {}

def rodar(nome, fn):
    try:
        fn()
        print(f"[ok] {nome}")
    except Exception as e:
        FALHAS[nome] = f"{type(e).__name__}: {e}"
        print(f"[FALHOU] {nome}: {type(e).__name__}: {e}")
        traceback.print_exc()


## STab (implementação própria — ver `src/churn_telecom/models/stab.py`)

Não existe um pacote público legítimo chamado "STab" (colisão de nome no
PyPI — ver smoke test). Esta é uma implementação própria e enxuta de
Transformer para dados tabulares, cobrindo o espaço de hiperparâmetros
pedido no enunciado.

In [ ]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

def experimento_stab():
    hp = STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1,
                         lr=1e-3, weight_decay=1e-5, batch_size=64, max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_stab(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    metricas = full_report(y_test, prever_stab(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "stab_churn", framework="PyTorch (STab — implementacao propria)",
             hyperparameters=hp.__dict__, metrics=metricas,
             architecture={"dim": hp.dim, "depth": hp.depth, "heads": hp.heads},
             history=historico, notes="STab (implementacao propria).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("stab", experimento_stab)


## TabPFN v2

A versão é fixada em **v2** (a do enunciado). Versões novas do pacote usam
por padrão checkpoints v2.5+, que exigem aceitar licença e um
`TABPFN_TOKEN` — inviável num "Save & Run All". Os pesos v2 baixam direto.

In [ ]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

def experimento_tabpfn():
    hp = TabPFNHyperparams(device=device, n_estimators=8, versao="v2", seed=SEED)
    modelo = treinar_tabpfn(X_train, y_train, hp)
    metricas = full_report(y_test, prever_tabpfn(modelo, X_test))
    print(metricas)
    save_run(RESULTS_DIR, "tabpfn_churn", framework="TabPFN v2", hyperparameters=hp.__dict__,
             metrics=metricas, notes="TabPFN v2 (checkpoint v2, sem token).",
             save_model_fn=lambda d: joblib.dump(modelo, d / "model.joblib"))

rodar("tabpfn", experimento_tabpfn)


## KAN (pykan)

In [ ]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

def experimento_kan():
    hp = KANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-2, steps=200, seed=SEED)
    modelo, historico = treinar_kan(X_train, y_train, X_val, y_val, hp, device=device)
    metricas = full_report(y_test, prever_kan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "kan_churn", framework="pykan (KAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "grid": hp.grid, "k": hp.k},
             history=pd.DataFrame(historico), notes="KAN.",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("kan", experimento_kan)


## TabKAN (ChebyshevKAN do pacote `tabkan`)

O pacote `tabkan` expõe módulos PyTorch (`ChebyshevKAN`, `FourierKAN`, ...),
não um classificador sklearn-like — o wrapper usa `ChebyshevKAN` com
treino Adam + early stopping na validação, igual ao STab.

In [ ]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

def experimento_tabkan():
    hp = TabKANHyperparams(hidden_width=10, degree=3, lr=1e-3, batch_size=64,
                           max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_tabkan(X_train, y_train, X_val, y_val, hp, device=device)
    metricas = full_report(y_test, prever_tabkan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "tabkan_churn", framework="tabkan (ChebyshevKAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "degree": hp.degree},
             history=historico, notes="TabKAN (ChebyshevKAN).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("tabkan", experimento_tabkan)


## Mitra (via AutoGluon)

In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra, salvar_mitra

def experimento_mitra():
    hp = MitraHyperparams(time_limit_segundos=600, num_gpus=1 if device == "cuda" else 0, seed=SEED)
    preditor = treinar_mitra(X_train, y_train, cols, hp, path="/kaggle/working/ag_mitra")
    metricas = full_report(y_test, prever_mitra(preditor, X_test, cols))
    print(metricas)
    save_run(RESULTS_DIR, "mitra_churn", framework="AutoGluon (extra [mitra])",
             hyperparameters=hp.__dict__, metrics=metricas, notes="Mitra.",
             save_model_fn=lambda d: salvar_mitra(preditor, d / "autogluon_predictor"))

rodar("mitra", experimento_mitra)


## Comparação final — todos os modelos do projeto

Funciona mesmo que algum modelo acima tenha falhado.

In [ ]:
pd.DataFrame([{"modelo": k, "erro": v} for k, v in FALHAS.items()],
             columns=["modelo", "erro"]).to_csv(TABLES_DIR / "modelos_avancados_falhas_kaggle.csv", index=False)
print("Falhas:", FALHAS or "nenhuma")

resumo = load_all_metadata(RESULTS_DIR)
cols_mostrar = ["model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
cols_existentes = [c for c in cols_mostrar if c in resumo.columns]
tabela = resumo[cols_existentes].sort_values("metrics.ks", ascending=False).reset_index(drop=True)
tabela


## Trazendo os resultados de volta

O "Save & Run All (Commit)" do Kaggle versiona os arquivos gerados em
`/kaggle/working`. Depois, baixe a pasta `results/` do commit e cole em
`results/` no projeto local, junto com qualquer CSV novo em
`reports/tables/` — o notebook de consolidação final só lê `results/`,
não retreina nada.
